<a href="https://colab.research.google.com/github/KarlaMichelleSorianoSanhez/Procesos-estocasticos/blob/main/Metodo_de_uniformizaci%C3%B3n.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1><font color="#002060">MÉTODO DE UNIFORMIZACIÓN PARA CADENAS DE MARKOV EN TIEMPO CONTINUO</font></h3>

**Nombre:** Karla Michelle Soriano Sánchez


**Objetivo** : Implementar el método de uniformización para aproximar la matriz de probabilidades de transición de una cadena de Markov en tiempo continuo y verificar numéricamente la ecuación de Chapman-Kolmogorov.



<b>Teorema Matriz $P(t)$):</b>
La matriz de probabilidad de transición

$$
P(t)=\big[p_{ij}(t)\big]
$$

está dada por

$$
P(t)
=
\sum_{k=0}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}
\hat P^{\,k}.
$$

Este resultado permite expresar la matriz de transición de una cadena de Markov en tiempo continuo mediante una combinación ponderada de las potencias de la matriz estocástica $\hat P$

<b>Aproximación numérica:</b>

Como la serie anterior es infinita, para efectos computacionales se aproxima utilizando únicamente los primeros $M$ términos.

Una elección adecuada para el truncamiento es

$$
M
\approx
\max\{rt+5\sqrt{rt},20\}.
$$

Con este valor se garantiza que los términos omitidos tengan una contribución despreciable y la aproximación obtenida sea suficientemente precisa e la matriz $P(t)$.


<h2><font color="#1E4FA1">EJERCICIO 3.</font></h2>


Sea la matriz

$$
R=
\begin{pmatrix}
0 & 2 & 3 & 0\\
4 & 0 & 2 & 0\\
0 & 2 & 0 & 2\\
1 & 0 & 3 & 0
\end{pmatrix}.
$$

1. Use esta propuesta para calcular $P(0.5)$, $P(1)$ y $P(5)$.

2. ¿Se verifica la ecuación de Chapman-Kolmogorov?

$$
P(1)=P(0.5)P(0.5)
$$

### Importación de librerías

Utilizaremos la biblioteca SymPy para realizar los cálculos matriciales de manera simbólica y numérica.

Esto permitirá implementar directamente las expresiones obtenidas en los teoremas demostrados anteriormente.

In [2]:
import sympy as sp

### Definición de la matriz de tasas

La matriz $R=[r_{ij}]$ describe la dinámica de la cadena de Markov en tiempo continuo.

Cada elemento $r_{ij}$ representa la tasa con la que el proceso transita del estado $i$ al estado $j$.

A partir de esta matriz construiremos la matriz estocástica $\hat P requerida por el método de uniformización.

In [3]:
R = sp.Matrix([
    [0, 2, 3, 0],
    [4, 0, 2, 0],
    [0, 2, 0, 2],
    [1, 0, 3, 0]
])

R

Matrix([
[0, 2, 3, 0],
[4, 0, 2, 0],
[0, 2, 0, 2],
[1, 0, 3, 0]])

### Cálculo del parámetro r

El método de uniformización requiere seleccionar un número

$$
r \geq \max_i\{r_i\},
$$

donde

$$
r_i=\sum_{j=1}^{N}r_{ij}.
$$

Como el teorema exige un número

$$
r \geq \max_i\{r_i\},
$$

calculamos primero las sumas por renglón

$$
r_i=\sum_j r_{ij}
$$

y elegimos el mayor valor.

In [4]:
def calcular_r(R):
  #sumamos fila a fila para obtener r max
  r_i = [sum(fila) for fila in R.tolist()]

  # Elegimos el valor máximo
  r = max(r_i)

  return r_i, r

In [5]:
r_i, r = calcular_r(R)

print(f"Sumas Individuales (r_i): {r_i}")
print(f"Maxima valor de la suma (r): {r}")

Sumas Individuales (r_i): [5, 6, 4, 4]
Maxima valor de la suma (r): 6


### Construcción de la matriz estocástica $\hat P$

De acuerdo con el teorema de uniformización,

$$
\hat p_{ij}
=
\begin{cases}
1-\dfrac{r_i}{r}, & i=j,\\
\dfrac{r_{ij}}{r}, & i\neq j.
\end{cases}
$$

La matriz $\hat P$ será la matriz de transición de la cadena embebida.

In [6]:
def construir_P_hat(R, r):
  n = R.shape[0]
  P_hat = sp.zeros(n)
  r_i = [sum(fila) for fila in R.tolist()]

  for i in range(n):
    for j in range(n):
      if i == j:
        P_hat[i, j] = 1 - r_i[i]/r

      else:
        P_hat[i, j] = R[i, j]/r

  return sp.Matrix(P_hat)

In [7]:
P_hat = construir_P_hat(R, r)

print("Matriz P_hat:")
display(P_hat)

Matriz P_hat:


Matrix([
[1/6, 1/3, 1/2,   0],
[2/3,   0, 1/3,   0],
[  0, 1/3, 1/3, 1/3],
[1/6,   0, 1/2, 1/3]])

### Verificación de la propiedad estocástica

Antes de continuar verificamos que cada renglón de $\hat P$ sume uno.

Esto garantiza que $\hat P$ es una matriz de transición *válida*.

In [8]:
for i in range(P_hat.rows):
  suma = sum(P_hat[i, j] for j in range(P_hat.cols))
  print(f"Fila {i+1}: {sp.simplify(suma)}")

Fila 1: 1
Fila 2: 1
Fila 3: 1
Fila 4: 1


### Número de términos de truncamiento

El ejercicio propone aproximar la serie infinita utilizando

$$
M \approx \max\{rt+5\sqrt{rt},20\}.
$$

Este valor permite truncar la serie conservando una buena aproximación de $P(t)$.

In [9]:
def calcular_M(r, t):
  rt = r*t
  M = int(sp.ceiling(max(rt + 5*sp.sqrt(rt), 20)))
  return M

### Implementación del Teorema de Uniformización

La matriz de transición está dada por

$$
P(t)
=
\sum_{k=0}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}
\hat P^k.
$$


Cada término

$$
e^{-rt}\frac{(rt)^k}{k!}
$$

corresponde a la probabilidad de que un proceso de Poisson de tasa r
presente exactamente k eventos en el intervalo [0,t].



Como no es posible calcular infinitos términos, utilizamos el truncamiento definido anteriormente y aproximamos

$$
P(t)
\approx
\sum_{k=0}^{M}
e^{-rt}
\frac{(rt)^k}{k!}
\hat P^k.
$$

La siguiente función implementa directamente esta expresión.

In [10]:
def calcular_P(t, P_hat, r):
  M = calcular_M(r, t)
  print(f"\nPara t = {t}")
  print(f"M = {M}")

  n = P_hat.shape[0]
  P = sp.zeros(n)

  for k in range(M + 1):
    coeficiente = sp.exp(-r*t) * (r*t)**k / sp.factorial(k)
    P += coeficiente * (P_hat**k)

  return sp.N(P, 12)

### Cálculo de las matrices de transición

Aplicamos el método de uniformización para los tiempos solicitados en el problema.

In [11]:
P05 = calcular_P(0.5, P_hat, r)

print("\nP(0.5)")
display(P05)

P1 = calcular_P(1, P_hat, r)

print("\nP(1)")
display(P1)

P5 = calcular_P(5, P_hat, r)

print("\nP(5)")
display(P5)


Para t = 0.5
M = 20

P(0.5)


Matrix([
[0.250608679645, 0.216964598159, 0.386656935582, 0.145769786602],
[0.253134844845, 0.238360983781, 0.374409239718, 0.134094931645],
[0.169119496516, 0.193614888245, 0.420301017069, 0.216964598159],
[0.158017482467, 0.157444641559, 0.398331790539, 0.286206085423]])


Para t = 1
M = 20

P(1)


Matrix([
[0.206151120157, 0.203902026694, 0.398709595696, 0.191235802346],
[ 0.20828421449, 0.205340699017, 0.397899174557, 0.188474456828],
[ 0.19675849338, 0.198379335659,  0.40095868916, 0.203902026694],
[0.192046223485, 0.193997147863, 0.401470941214, 0.212484232331]])


Para t = 5
M = 58

P(5)


Matrix([
[ 0.19999962635, 0.199999625709,  0.39999924811, 0.199999621199],
[0.199999627061,   0.1999996262,  0.39999924796, 0.199999620146],
[0.199999623304, 0.199999623603, 0.399999248751, 0.199999625709],
[0.199999621348, 0.199999622251, 0.399999249163, 0.199999628605]])

Observamos que para t=5 todas las filas son prácticamente iguales.
Esto indica que la cadena se encuentra cercana a su comportamiento
estacionario y la distribución de probabilidad ya no depende de forma
significativa del estado inicial.

### Verificación de Chapman-Kolmogorov

Para una cadena de Markov en tiempo continuo se cumple

$$
P(t+s)=P(t)P(s).
$$

Tomando

$$
t=s=0.5,
$$

debemos verificar que

$$
P(1)=P(0.5)P(0.5).
$$

In [12]:
producto = P05 * P05

print("P(0.5)P(0.5)")
display(sp.N(producto, 12))

print("P(1)")
display(P1)

P(0.5)P(0.5)


Matrix([
[0.206151411174, 0.203902317711,  0.39871017773, 0.191236093362],
[0.208284505507, 0.205340990034,  0.39789975659, 0.188474747845],
[0.196758784397, 0.198379626676, 0.400959271193, 0.203902317711],
[0.192046514502,  0.19399743888, 0.401471523247, 0.212484523348]])

P(1)


Matrix([
[0.206151120157, 0.203902026694, 0.398709595696, 0.191235802346],
[ 0.20828421449, 0.205340699017, 0.397899174557, 0.188474456828],
[ 0.19675849338, 0.198379335659,  0.40095868916, 0.203902026694],
[0.192046223485, 0.193997147863, 0.401470941214, 0.212484232331]])

### Cálculo del error

Finalmente calculamos la diferencia entre ambas matrices.

Si el error es suficientemente pequeño, podremos concluir que la ecuación de Chapman-Kolmogorov se verifica numéricamente.

In [13]:
error = P1 - producto

print("Error:")
display(sp.N(error, 12))

error_max = max(
    abs(float(error[i, j]))
    for i in range(error.rows)
    for j in range(error.cols)
)

print("Error máximo =", error_max)

Error:


Matrix([
[-2.91016675646e-7, -2.91016704068e-7, -5.82033351293e-7, -2.91016647225e-7],
[-2.91016675646e-7, -2.91016704068e-7, -5.82033351293e-7, -2.91016675646e-7],
[-2.91016647225e-7, -2.91016704068e-7, -5.82033408136e-7, -2.91016647225e-7],
[-2.91016675646e-7, -2.91016675646e-7, -5.82033351293e-7, -2.91016675646e-7]])

Error máximo = 5.820334081363399e-07


### Conclusión

Se implementó exitosamente el método de uniformización para aproximar la matriz de transición de una cadena de Markov en tiempo continuo.

Se calcularon las matrices $P(0.5)$, $P(1)$ y $P(5)$ utilizando el truncamiento sugerido por el ejercicio.

Además, la verificación de Chapman-Kolmogorov mostró un error numérico muy pequeño, confirmando la validez del procedimiento implementado y la consistencia de los resultados obtenidos.

<h3><font color="#1E4FA1">Ejercicio 4</font></h3>

<b>Teorema (Cotas de error para $P(t)$):</b>

Para un $t\geq 0$ fijo, sea

$$
P^{M}(t)=\left[p^{M}_{i,j}(t)\right]
=
\sum_{k=0}^{M}
e^{-rt}
\frac{(rt)^k}{k!}
\hat P^{\,k}
$$

entonces

$$
\left|p_{i,j}(t)-p^{M}_{i,j}(t)\right|
\leq
\sum_{k=M+1}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}
$$

para todo

$$
1\leq i,j\leq N.
$$

Además, para calcular $P(t)$ con una tolerancia $\varepsilon$, se debe elegir $M$ tal que

$$
\sum_{k=M+1}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}
\leq \varepsilon.
$$



<br>

<b>Ejercicio para programar.</b>

Este teorema se puede usar así:

Suponga que se desea calcular $P(t)$ con una tolerancia $\varepsilon$.

Elija $M$ tal que

$$
\sum_{k=M+1}^{\infty}
e^{-rt}
\frac{(rt)^k}{k!}
\leq \varepsilon.
$$

Y se puede implementar de acuerdo al siguiente **algoritmo de uniformización para $P(t)$**:

1. Dados $R,t, 0<\varepsilon<1$.

2. Calcular $r$ usando la igualdad en la definición.

3. Calcular $\hat P$.

4.

$$
A=\hat P;
\qquad
B=e^{-rt}I;
\qquad
c=e^{-rt};
\qquad
sum=c;
\qquad
k=1
$$

5. Mientras $sum<1-\varepsilon$ hacer:

$$
c=c\frac{rt}{k}
$$

$$
B=B+cA
$$

$$
A=A\hat P
$$

$$
sum=sum+c
$$

$$
k=k+1
$$

6. $B$ está a $\varepsilon$ de $P(t)$.

<br>

Repita el ejercicio 3 aplicando este algoritmo con una tolerancia

$$
\varepsilon = 0.00001
$$

(indique el valor correspondiente de $M$ en cada caso).

Compare los resultados.



En el ejercicio anterior aproximamos la matriz de transición utilizando
una regla práctica para determinar el número de términos de la serie.

Ahora emplearemos una estrategia más rigurosa basada en la cota del error
de truncamiento.

La idea consiste en agregar términos de la serie de uniformización hasta
que la probabilidad acumulada sea suficientemente cercana a uno.

De esta forma garantizamos que el error de truncamiento sea menor que una tolerancia previamente especificada.

La tolerancia especificada en el problema es

$$
\varepsilon = 10^{-5}.
$$

Reutilizaremos la matriz de tasas $R$, el parámetro $r$ y la matriz
estocástica $\hat P$ obtenidos en el Ejercicio 3.

In [1]:
# ==========================================================
# Función que implementa el algoritmo de uniformización
# utilizando una tolerancia epsilon
# ==========================================================

def uniformizacion_tolerancia(t, P_hat, r, epsilon=1e-5):

  # Calculamos rt
  rt = r * t

  # Número de estados de la cadena
  n = P_hat.rows

  # Inicialización
  A = P_hat.copy()

  B = sp.exp(-rt) * sp.eye(n)

  c = sp.exp(-rt)

  suma = float(c)

  k = 1

  # Se agregan términos hasta alcanzar la tolerancia

  """
  Mientras la probabilidad acumulada sea menor que 1-epsilon
  seguimos agregando términos de la serie

  """
  while suma < 1 - epsilon:
    # Nuevo coeficiente de Poisson
    c = c * rt / k

    # Actualizamos aproximación de P(t)
    B = B + c * A

    # Calculamos la siguiente potencia de P_hat
    A = A * P_hat

    # Actualizamos probabilidad acumulada
    suma += float(c)

    k += 1

    # Último término utilizado
    M = k - 1

  return sp.N(B,12), M

Aplicaremos el algoritmo para los mismos tiempos considerados en el
Ejercicio 3:

$$
t=0.5,\qquad t=1,\qquad t=5.
$$

Además de calcular las matrices de transición, registraremos el valor
de $M$ utilizado en cada caso.

Este valor representa el número mínimo de términos necesarios para
garantizar que el error de trucamiento sea menor que la tolerancia especificada.



In [18]:
# Cálculo de P(t) para los tiempos solicitados
P05_tol, M05 = uniformizacion_tolerancia(0.5, P_hat, r)

P1_tol, M1 = uniformizacion_tolerancia(1, P_hat, r)

P5_tol, M5 = uniformizacion_tolerancia(5, P_hat, r)

# Mostrar matrices obtenidas
print("P(0.5)")
display(P05_tol)

print("\nP(1)")
display(P1_tol)

print("\nP(5)")
display(P5_tol)

P(0.5)


Matrix([
[0.250607999264, 0.216963917777, 0.386655574821, 0.145769106223],
[0.253134164463, 0.238360303399, 0.374407878957, 0.134094251267],
[0.169118816135, 0.193614207865, 0.420299656308, 0.216963917777],
[0.158016802088, 0.157443961179, 0.398330429778, 0.286205405041]])


P(1)


Matrix([
[0.206150375145, 0.203901281681, 0.398708105672, 0.191235057333],
[0.208283469478, 0.205339954005, 0.397897684532, 0.188473711816],
[0.196757748368, 0.198378590647, 0.400957199135, 0.203901281681],
[0.192045478473, 0.193996402851, 0.401469451189, 0.212483487319]])


P(5)


Matrix([
[0.199998526286, 0.199998525644, 0.399997047981, 0.199998521134],
[0.199998526996, 0.199998526136, 0.399997047831, 0.199998520081],
[ 0.19999852324, 0.199998523539, 0.399997048622, 0.199998525644],
[0.199998521284, 0.199998522187, 0.399997049034, 0.199998528541]])



El problema solicita indicar explícitamente el valor de $M$ utilizado
en cada cálculo.

Por ello mostramos un resumen de los valores obtenidos por el algoritmo.
Obsérvese que conforme aumenta el tiempo $t$, también aumenta el número de términos necesarios para aproximar adecuadamente la serie de uniformización.

In [15]:
# Valores de M obtenidos

print("Resumen de valores de M\n")

print(f"t = 0.5  --->  M = {M05}")

print(f"t = 1    --->  M = {M1}")

print(f"t = 5    --->  M = {M5}")

Resumen de valores de M

t = 0.5  --->  M = 13
t = 1    --->  M = 19
t = 5    --->  M = 56




Finalmente compararemos las matrices obtenidas mediante el criterio de
tolerancia con las matrices calculadas en el Ejercicio 3.

Si las diferencias son muy pequeñas podremos concluir que ambos métodos
producen esencialmente la misma aproximación de la matriz de transición.

In [16]:
# Diferencias entre ambos métodos
print("Diferencia para t = 0.5")
display(sp.N(P05_tol - P05,12))

print("Diferencia para t = 1")
display(sp.N(P1_tol - P1,12))

print("Diferencia para t = 5")
display(sp.N(P5_tol - P5,12))

Diferencia para t = 0.5


Matrix([
[-6.80381617713e-7, -6.80381191387e-7,   -1.360760848e-6, -6.80379201867e-7],
[-6.80381560869e-7, -6.80381674556e-7, -1.36076096169e-6, -6.80378633433e-7],
[-6.80380281892e-7, -6.80380111362e-7, -1.36076124591e-6, -6.80381191387e-7],
[ -6.8037908818e-7, -6.80379713458e-7, -1.36076141644e-6, -6.80382584051e-7]])

Diferencia para t = 1


Matrix([
[-7.45012414427e-7, -7.45012386005e-7, -1.49002477201e-6, -7.45012386005e-7],
[-7.45012414427e-7, -7.45012386005e-7, -1.49002477201e-6, -7.45012386005e-7],
[-7.45012414427e-7, -7.45012386005e-7, -1.49002477201e-6, -7.45012386005e-7],
[-7.45012386005e-7, -7.45012386005e-7, -1.49002477201e-6, -7.45012386005e-7]])

Diferencia para t = 5


Matrix([
[-1.1000644804e-6, -1.1000644804e-6, -2.20012896079e-6, -1.1000644804e-6],
[-1.1000644804e-6, -1.1000644804e-6, -2.20012896079e-6, -1.1000644804e-6],
[-1.1000644804e-6, -1.1000644804e-6, -2.20012896079e-6, -1.1000644804e-6],
[-1.1000644804e-6, -1.1000644804e-6, -2.20012896079e-6, -1.1000644804e-6]])



Para cuantificar la diferencia entre ambos procedimientos calculamos el
error absoluto máximo entre las matrices obtenidas.

Si dicho error es muy pequeño podremos afirmar que ambas aproximaciones
son prácticamente equivalentes.


In [17]:
# Error máximo para cada tiempo

error_05 = max(
    abs(float((P05_tol - P05)[i,j]))
    for i in range(P05.rows)
    for j in range(P05.cols)
)

error_1 = max(
    abs(float((P1_tol - P1)[i,j]))
    for i in range(P1.rows)
    for j in range(P1.cols)
)

error_5 = max(
    abs(float((P5_tol - P5)[i,j]))
    for i in range(P5.rows)
    for j in range(P5.cols)
)

print("Error máximo para t = 0.5 :", error_05)

print("Error máximo para t = 1   :", error_1)

print("Error máximo para t = 5   :", error_5)

Error máximo para t = 0.5 : 1.3607614164357074e-06
Error máximo para t = 1   : 1.4900247720106563e-06
Error máximo para t = 5   : 2.2001289607942454e-06


Se implementó correctamente el algoritmo de uniformización basado en la cota teórica de error del truncamiento utilizando una tolerancia

$$
\varepsilon = 10^{-5}.
$$

A diferencia del Ejercicio 3, donde el número de términos se seleccionó mediante una regla práctica, en este ejercicio se empleó un criterio basado directamente en la cota del error, lo que permitió determinar automáticamente el número mínimo de términos necesarios para alcanzar la precisión requerida.

Aplicando este procedimiento se calcularon las matrices de transición

$$
P(0.5), \qquad P(1), \qquad P(5),
$$

obteniéndose los siguientes valores de truncamiento:

$$
M(0.5)=13,
\qquad
M(1)=19,
\qquad
M(5)=56.
$$

Posteriormente, los resultados fueron comparados con las matrices obtenidas en el Ejercicio 3 mediante la aproximación

$$
M \approx \max{rt+5\sqrt{rt},20}.
$$

La diferencia entre ambos procedimientos resultó ser del orden de

$$
10^{-6},
$$

valor considerablemente menor que la tolerancia especificada. Esto indica que las matrices calculadas por ambos métodos son prácticamente idénticas.

Por lo tanto, se concluye que el algoritmo de uniformización fue implementado correctamente y que la cota teórica de error proporciona un criterio confiable para seleccionar el número de términos de la serie. Además, la comparación con el Ejercicio 3 confirma la consistencia de los resultados obtenidos y demuestra que ambos enfoques generan aproximaciones altamente precisas de la matriz de transición $P(t)$.
